# Fine-Tune Pretrained Commutative CNN Classifier

Load the pretrained commutative CNN encoder and fine-tune the full network on the current labeled action dataset.

In [ ]:
%load_ext autoreload
%autoreload 2

from dataclasses import asdict
from pathlib import Path

import pandas as pd

from src.ml import (
    CommutativeCNNClassifier,
    LossWeightConfig,
    OptimizationConfig,
    create_experiment_run,
    display_experiment_summary,
    display_holdout_evaluation,
    fit_chunked_water_vs_other_hot_start,
    fit_estimator_on_experiment,
    load_commutative_cnn_pretraining_config,
    persist_experiment_artifacts,
    plot_training_history,
    prepare_multitask_experiment_data,
    prepare_water_vs_other_pretraining_data,
)
from src.dataset_config import load_current_dataset_artifact_path
from src.tensor_utils import (
    build_tensor_embedding_2d,
    load_labeled_tensor_dataset,
    plot_tensor_embedding_2d,
)

pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)
pd.set_option("display.width", None)
pd.set_option("display.expand_frame_repr", False)


In [ ]:
# User inputs

dataset_artifact_path = load_current_dataset_artifact_path()
unlabeled_dataset_path = Path(".dataset_cache/unlabeled_active_high_mid_low_t20_z5_y96_x96_chunks")
pretraining_config_path = Path("artifacts/pretrained_commutative_cnn/config.yaml")
pretraining_config = load_commutative_cnn_pretraining_config(pretraining_config_path)
print(f"Loaded commutative CNN pretraining config from {pretraining_config_path}")
print(pretraining_config)

pretrained_encoder_path = pretraining_config.pretrained_encoder_path
if not pretrained_encoder_path.exists():
    raise FileNotFoundError(
        f"Pretrained CNN encoder not found at {pretrained_encoder_path}. "
        "Run 10C_pretrain_commutative_cnn_encoder.ipynb first, or update "
        "artifacts/pretrained_commutative_cnn/config.yaml to point at an existing checkpoint."
    )
model_config = pretraining_config.model_config
experiment_output_dir = Path("artifacts/nb13C_commutative_cnn_full_finetune")
experiment_run = create_experiment_run(experiment_output_dir, "13C_finetune_commutative_cnn")
loss_plot_root = Path(experiment_run.loss_plot_dir)
figure_dir = Path(experiment_run.figure_dir)
binary_loss_plot_dir = loss_plot_root / "binary_hot_start"
fine_tune_loss_plot_dir = loss_plot_root / "fine_tune"
persist_artifacts = True
print(f"Experiment id: {experiment_run.experiment_id}")
print(f"Experiment run folder: {Path(experiment_run.run_dir).resolve()}")
print(f"Using pretrained encoder checkpoint: {pretrained_encoder_path.resolve()}")
print(f"Binary hot-start loss PDFs: {binary_loss_plot_dir.resolve()}")
print(f"Fine-tune loss PDFs: {fine_tune_loss_plot_dir.resolve()}")

holdout_fraction = 0.25
validation_fraction_within_train = 0.20
train_num_random_rotations = 6
rotation_range_degrees = 12.0
run_binary_hot_start = False
binary_pretraining_epochs = 2
binary_learning_rate = 1e-5
binary_weight_decay = 7.5e-4
binary_class_weighting = None

optimization_config = OptimizationConfig(
    batch_size=8,
    epochs=90,
    learning_rate=1e-5,
    weight_decay=1e-3,
    early_stopping_patience=10,
    early_stopping_min_delta=5e-5,
    early_stopping_start_epoch=10,
    early_stopping_monitor="action_loss",
    early_stopping_smoothing="median",
    early_stopping_smoothing_window=5,
    scheduler_patience=3,
    scheduler_factor=0.5,
    scheduler_min_lr=1e-6,
    training_plot_dir=str(fine_tune_loss_plot_dir),
    training_plot_every_n_epochs=1,
    validation_split=0.0,
    random_state=0,
    standardize=True,
    device=None,
    verbose=True,
    class_weighting=None,
)
loss_weight_config = LossWeightConfig(
    action_weight=1.0,
    water_vs_other_weight=0.075,
    compound_weight=0.03,
    concentration_weight=0.03,
    latent_alignment_weight=0.0,
    lambda_align=0.0,
)


## Output Review

The previous 13C/v9 fine-tune improved action ranking and drug-only separation, but the final decision rule was badly calibrated: Water recall collapsed and AChE absorbed many holdout samples. Compound and concentration reports were also not meaningful because those heads were reported while their supervised weights were zero.

This next run removes the binary hot-start by default (`run_binary_hot_start=False`), keeps only a small explicit water boundary (`water_vs_other_weight=0.075`), disables balanced class weighting, and trains the auxiliary compound/concentration heads lightly (`0.03` each). The goal is to test whether multiclass calibration improves without losing the drug-only action separation seen in v9.

In [ ]:
dataset = load_labeled_tensor_dataset(dataset_artifact_path)
experiment = prepare_multitask_experiment_data(
    dataset,
    holdout_fraction=holdout_fraction,
    validation_fraction_within_train=validation_fraction_within_train,
    train_num_random_rotations=train_num_random_rotations,
    rotation_range_degrees=rotation_range_degrees,
    random_state=optimization_config.random_state,
)
display_experiment_summary(experiment)

## Memory Note

The unlabeled pretraining chunks are large and `load_unlabeled_tensor_dataset(...)` concatenates them into one CPU tensor. This notebook keeps the chunked binary hot-start code path available for controlled ablations, but the next run disables it by default so the main test is multiclass calibration with a small supervised water boundary.

In [ ]:
model = CommutativeCNNClassifier(
    model_config=model_config,
    optimization_config=optimization_config,
    loss_weight_config=loss_weight_config,
    pretrained_state_path=pretrained_encoder_path,
    freeze_backbone=False,
    hot_start=True,
)

binary_pretraining_data = None
binary_pretraining_history = None
if run_binary_hot_start and binary_pretraining_epochs > 0:
    final_epochs = model.epochs
    final_learning_rate = model.learning_rate
    final_weight_decay = model.weight_decay
    model.training_plot_dir = str(binary_loss_plot_dir)
    model.training_plot_title = "Water-vs-other hot-start loss curves"
    model.learning_rate = binary_learning_rate
    model.weight_decay = binary_weight_decay
    print(f"Writing binary hot-start loss PDFs to: {binary_loss_plot_dir.resolve()}")
    print(f"Binary hot-start optimizer: lr={model.learning_rate:g}, weight_decay={model.weight_decay:g}, class_weighting={binary_class_weighting}")
    binary_pretraining_data = fit_chunked_water_vs_other_hot_start(
        model,
        unlabeled_dataset_path,
        holdout_metadata=experiment.splits.metadata_holdout,
        validation_fraction=validation_fraction_within_train,
        epochs=binary_pretraining_epochs,
        random_state=optimization_config.random_state,
        class_weighting=binary_class_weighting,
    )
    binary_pretraining_history = model.history_.copy()
    plot_training_history(model, title="Water-vs-other hot-start phase loss curves", loess_frac=0.6);
    model.epochs = final_epochs
    model.learning_rate = final_learning_rate
    model.weight_decay = final_weight_decay

model.training_plot_dir = str(fine_tune_loss_plot_dir)
model.training_plot_title = "Pretrained commutative CNN full fine-tune loss curves"
print(f"Writing fine-tune loss PDFs to: {fine_tune_loss_plot_dir.resolve()}")
fit_estimator_on_experiment(model, experiment)
plot_training_history(model, title="Pretrained commutative CNN full fine-tune loss curves", loess_frac=0.6);


In [ ]:
holdout_evaluation = display_holdout_evaluation(model, experiment)

In [ ]:
holdout_embedding_projection = build_tensor_embedding_2d(
    model.transform(experiment.splits.X_holdout),
    experiment.y_true_holdout["action"],
    label_map=experiment.label_maps["action"],
    metadata=experiment.splits.metadata_holdout,
    method="umap",
    random_state=optimization_config.random_state,
)
holdout_embedding_projection.to_csv(
    figure_dir / f"{experiment_run.experiment_id}_holdout_embedding_umap.csv",
    index=False,
)
plot_tensor_embedding_2d(
    holdout_embedding_projection,
    title="Holdout embedding projection by action",
    marker_column="compound",
    output_path=figure_dir / f"{experiment_run.experiment_id}_holdout_embedding_umap.pdf",
)

In [ ]:
import torch

all_labeled_tensors = torch.cat(
    [
        experiment.splits.X_train_base,
        experiment.splits.X_val,
        experiment.splits.X_holdout,
    ],
    dim=0,
)
all_action_labels = torch.cat(
    [
        experiment.splits.y_train_base,
        torch.as_tensor(experiment.splits.y_val),
        torch.as_tensor(experiment.splits.y_holdout),
    ]
).numpy()
all_labeled_metadata = pd.concat(
    [
        experiment.splits.metadata_train_base.assign(dataset_split="train"),
        experiment.splits.metadata_val.assign(dataset_split="validation"),
        experiment.splits.metadata_holdout.assign(dataset_split="holdout"),
    ],
    ignore_index=True,
)

all_embedding_projection = build_tensor_embedding_2d(
    model.transform(all_labeled_tensors),
    all_action_labels,
    label_map=experiment.label_maps["action"],
    metadata=all_labeled_metadata,
    method="umap",
    random_state=optimization_config.random_state,
)
all_embedding_projection.to_csv(
    figure_dir / f"{experiment_run.experiment_id}_all_labeled_embedding_umap.csv",
    index=False,
)
plot_tensor_embedding_2d(
    all_embedding_projection,
    title="All labeled data embedding projection by action (with controls)",
    marker_column="compound",
    edge_color_column="dataset_split",
    edge_color_map={"train": "white", "validation": "black", "holdout": "black"},
    display_control=True,
    output_path=figure_dir / f"{experiment_run.experiment_id}_all_labeled_embedding_umap_with_controls.pdf",
)
plot_tensor_embedding_2d(
    all_embedding_projection,
    title="All labeled data embedding projection by action (controls hidden)",
    marker_column="compound",
    edge_color_column="dataset_split",
    edge_color_map={"train": "white", "validation": "black", "holdout": "black"},
    display_control=False,
    output_path=figure_dir / f"{experiment_run.experiment_id}_all_labeled_embedding_umap_controls_hidden.pdf",
)

In [ ]:
run_config = {
    "dataset_artifact_path": dataset_artifact_path,
    "experiment_id": experiment_run.experiment_id,
    "experiment_run_dir": Path(experiment_run.run_dir),
    "unlabeled_dataset_path": unlabeled_dataset_path,
    "pretraining_config_path": pretraining_config_path,
    "pretrained_encoder_path": pretrained_encoder_path,
    "binary_loss_plot_dir": binary_loss_plot_dir,
    "fine_tune_loss_plot_dir": fine_tune_loss_plot_dir,
    "freeze_backbone": False,
    "hot_start": True,
    "run_binary_hot_start": run_binary_hot_start,
    "binary_pretraining_epochs": binary_pretraining_epochs,
    "binary_learning_rate": binary_learning_rate,
    "binary_weight_decay": binary_weight_decay,
    "binary_class_weighting": binary_class_weighting,
    "binary_pretraining_excluded_holdout_count": None if binary_pretraining_data is None else binary_pretraining_data.excluded_holdout_count,
    "binary_pretraining_label_map": None if binary_pretraining_data is None else binary_pretraining_data.label_map,
    "binary_pretraining_train_count": None if binary_pretraining_data is None else binary_pretraining_data.train_count,
    "binary_pretraining_val_count": None if binary_pretraining_data is None else binary_pretraining_data.val_count,
    "holdout_fraction": holdout_fraction,
    "validation_fraction_within_train": validation_fraction_within_train,
    "train_num_random_rotations": train_num_random_rotations,
    "rotation_range_degrees": rotation_range_degrees,
    "model_config": asdict(model_config),
    "optimization_config": asdict(optimization_config),
    "loss_weight_config": asdict(loss_weight_config),
}
if persist_artifacts:
    experiment_artifacts = persist_experiment_artifacts(
        output_dir=experiment_output_dir,
        estimator=model,
        reports=holdout_evaluation.reports,
        config=run_config,
        experiment_prefix="13C_finetune_commutative_cnn",
        experiment_id=experiment_run.experiment_id,
        evaluation=holdout_evaluation,
        experiment=experiment,
        loss_plot_dirs=[binary_loss_plot_dir, fine_tune_loss_plot_dir],
        analysis=(
            "Planned calibration run: remove binary hot-start by default, use a small supervised water boundary, "
            "disable balanced class weighting, and train compound/concentration heads lightly so reported metrics "
            "match the objective. After the run, replace this with validation action loss, holdout action macro-F1, "
            "Water recall, and drug-only macro-F1."
        ),
        next_round_proposal=(
            "If Water recall recovers without losing drug-only macro-F1, sweep water_vs_other_weight around 0.05-0.10. "
            "If AChE still absorbs Water, keep class_weighting=None and reduce the water boundary; if drug recall drops, "
            "try a 2-epoch binary hot-start ablation."
        ),
    )
    if binary_pretraining_history is not None:
        binary_pretraining_history.to_csv(
            Path(experiment_run.run_dir) / f"{experiment_run.experiment_id}_binary_pretraining_history.csv",
            index=False,
        )
    experiment_artifacts
